In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:
SPLIT_BASE = Path("/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data")

# Hier werden die ESM-2-ready Dateien gespeichert
OUT_BASE = SPLIT_BASE / "ESM2"

OUT_BASE.mkdir(parents=True, exist_ok=True)

print("Split base:", SPLIT_BASE)
print("Output base:", OUT_BASE)

Split base: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data
Output base: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2


In [ ]:
TASKS = {
    "UBE4B": {
        "dir_candidates": ["UBE4B"],
        "indexing": "auto",
    },
    "GRB2": {
        "dir_candidates": ["GRB2"],
        "indexing": "auto",
    },
    "PTEN_activity": {
        "dir_candidates": ["PTEN_activity", "PTEN-activity", "pten_activity", "pten-activity"],
        "indexing": "auto",
    },
    "PTEN_abundance": {
        "dir_candidates": ["PTEN_abundance", "PTEN-abundance", "pten_abundance", "pten-abundance"],
        "indexing": "auto",
    },
    "Alpha-Amylase_expression": {
        "dir_candidates": [
            "Alpha-Amylase_expression",
            "Alpha-Amylase-expression",
            "alpha-amylase_expression",
            "alpha-amylase-expression",
            "AlphaAmylase_expression",
            "AlphaAmylase-expression",
            "Alpha_Amylase_expression",
        ],
        "indexing": "auto",
    },
    "Alpha-Amylase_thermostability": {
        "dir_candidates": [
            "Alpha-Amylase_thermostability",
            "Alpha-Amylase-thermostability",
            "alpha-amylase_thermostability",
            "alpha-amylase-thermostability",
            "AlphaAmylase_thermostability",
            "AlphaAmylase-thermostability",
            "Alpha_Amylase_thermostability",
        ],
        "indexing": "auto",
    },
    "Alpha-Amylase_specific_activity": {
        "dir_candidates": [
            "Alpha-Amylase_specific_activity",
            "Alpha-Amylase-specific_activity",
            "Alpha-Amylase-specific-activity",
            "alpha-amylase_specific_activity",
            "alpha-amylase-specific_activity",
            "alpha-amylase-specific-activity",
            "AlphaAmylase_specific_activity",
            "AlphaAmylase-specific_activity",
            "AlphaAmylase-specific-activity",
            "Alpha_Amylase_activity",
        ],
        "indexing": "auto",
    },
    "DLG4_abundance": {
        "dir_candidates": ["DLG4_abundance", "DLG4-abundance"],
        "indexing": "0-based",
    },
    "DLG4_binding": {
        "dir_candidates": ["DLG4_binding", "DLG4-binding"],
        "indexing": "0-based",
    },
}

In [ ]:
def find_existing_dir(base: Path, candidates):
    for name in candidates:
        path = base / name
        if path.exists() and path.is_dir():
            return path

    raise FileNotFoundError(
        "No matching split directory found. Tried:\n"
        + "\n".join(str(base / name) for name in candidates)
    )


def find_split_file(split_dir: Path, split: str):
    candidates = [
        split_dir / f"{split}.tsv",
        split_dir / f"{split}.csv",
        split_dir / f"{split}_data.tsv",
        split_dir / f"{split}_data.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find {split} file in {split_dir}. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


def read_table(path: Path):
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

In [ ]:
COLUMN_CANDIDATES = {
    "variant": ["variant", "mutant", "mutation", "name", "sample_id"],
    "sequence": ["sequence", "mutated_sequence"],
    "score": ["score", "DMS_score", "target", "label"],
}


def find_column(df: pd.DataFrame, candidates, required_name: str):
    cols = list(df.columns)

    for c in candidates:
        if c in cols:
            return c

    lower_map = {c.lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    raise ValueError(
        f"Could not find required column '{required_name}'. "
        f"Tried {candidates}. Available columns: {cols}"
    )


def normalize_split_df(df: pd.DataFrame, split: str) -> pd.DataFrame:
    variant_col = find_column(df, COLUMN_CANDIDATES["variant"], "variant")
    sequence_col = find_column(df, COLUMN_CANDIDATES["sequence"], "sequence")
    score_col = find_column(df, COLUMN_CANDIDATES["score"], "score")

    out = df[[variant_col, sequence_col, score_col]].copy()
    out.columns = ["variant", "sequence", "score"]
    out["split"] = split

    out["variant"] = out["variant"].astype(str).str.strip()
    out["sequence"] = out["sequence"].astype(str).str.strip()
    out["score"] = pd.to_numeric(out["score"], errors="coerce")

    before = len(out)
    out = out.dropna(subset=["variant", "sequence", "score"]).copy()
    out = out[out["sequence"].str.len() > 0].copy()

    removed = before - len(out)
    if removed > 0:
        print(f"[WARN] Removed {removed} invalid rows from {split} split.")

    return out

In [ ]:
def make_safe_sample_id(x: str) -> str:
    x = str(x).strip()
    x = x.replace("_", "-")
    x = x.replace(" ", "")
    x = x.replace("/", "-")
    x = x.replace(":", "-")
    x = x.replace(";", "-")
    return x


def parse_single_variant(variant: str):
    variant = str(variant).strip()

    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", variant)
    if match is None:
        raise ValueError(f"Could not parse single variant: {variant}")

    wt_aa = match.group(1)
    pos = int(match.group(2))
    mut_aa = match.group(3)

    return wt_aa, pos, mut_aa


def infer_or_apply_single_mutation(sequence: str, variant: str, indexing: str = "auto"):
    sequence = str(sequence).strip()
    wt_aa, pos, mut_aa = parse_single_variant(variant)

    if indexing == "1-based":
        candidate_indices = [pos - 1]
    elif indexing == "0-based":
        candidate_indices = [pos]
    elif indexing == "auto":
        candidate_indices = [pos - 1, pos]
    else:
        raise ValueError("indexing must be one of: 'auto', '1-based', '0-based'")

    observed = []

    for idx in candidate_indices:
        if not (0 <= idx < len(sequence)):
            observed.append((idx, None))
            continue

        aa_at_pos = sequence[idx]
        observed.append((idx, aa_at_pos))

        if aa_at_pos == wt_aa:
            seq_list = list(sequence)
            seq_list[idx] = mut_aa
            return "".join(seq_list), idx, "mutation_applied"
        if aa_at_pos == mut_aa:
            return sequence, idx, "already_mutated"

    raise ValueError(
        f"Could not match variant {variant} to sequence. "
        f"Expected WT={wt_aa} or MUT={mut_aa}. "
        f"Tried indices/observed: {observed}. "
        f"Sequence length: {len(sequence)}"
    )

In [ ]:
def prepare_esm2_from_existing_splits(
    task_name: str,
    split_dir: Path,
    out_dir: Path,
    indexing: str = "auto",
):
    print("\n" + "=" * 80)
    print(f"Preparing {task_name}")
    print("=" * 80)
    print("Split dir:", split_dir)
    print("Out dir:  ", out_dir)
    print("Indexing: ", indexing)

    out_dir.mkdir(parents=True, exist_ok=True)

    # Load splits
    split_tables = []

    for split in ["train", "val", "test"]:
        split_file = find_split_file(split_dir, split)
        df_raw = read_table(split_file)
        df = normalize_split_df(df_raw, split=split)
        split_tables.append(df)

        print(f"{split:5s}: {len(df):6d} rows loaded from {split_file.name}")

    full_df = pd.concat(split_tables, ignore_index=True)

    # Basic cleaning
    full_df["sample_id"] = full_df["variant"].map(make_safe_sample_id)

    before = len(full_df)
    full_df = full_df.dropna(subset=["sample_id", "sequence", "score"]).copy()
    full_df = full_df[full_df["sequence"].str.len() > 0].copy()
    after = len(full_df)

    if before != after:
        print(f"[WARN] Removed {before - after} rows after sample_id/sequence/score cleaning.")

    # Duplicate check
    dup_mask = full_df["sample_id"].duplicated(keep=False)

    if dup_mask.any():
        dup_path = out_dir / f"{task_name}_duplicate_sample_ids.tsv"
        full_df.loc[dup_mask].sort_values("sample_id").to_csv(dup_path, sep="\t", index=False)

        raise ValueError(
            f"{task_name}: duplicate sample_id values found. "
            f"Saved duplicates to: {dup_path}"
        )

    # Apply/infer mutated sequences
    used_sequences = []
    mutation_indices = []
    sequence_statuses = []
    failed_rows = []

    for row in full_df.itertuples(index=False):
        try:
            used_seq, idx, status = infer_or_apply_single_mutation(
                sequence=row.sequence,
                variant=row.variant,
                indexing=indexing,
            )
            used_sequences.append(used_seq)
            mutation_indices.append(idx)
            sequence_statuses.append(status)

        except Exception as e:
            used_sequences.append(np.nan)
            mutation_indices.append(np.nan)
            sequence_statuses.append("failed")
            failed_rows.append({
                "variant": row.variant,
                "sample_id": row.sample_id,
                "split": row.split,
                "error": str(e),
            })

    full_df["esm2_sequence"] = used_sequences
    full_df["mutation_index_0based"] = mutation_indices
    full_df["sequence_status"] = sequence_statuses

    if failed_rows:
        failed_df = pd.DataFrame(failed_rows)
        failed_path = out_dir / f"{task_name}_failed_mutation_application.tsv"
        failed_df.to_csv(failed_path, sep="\t", index=False)

        print(f"[ERROR] Failed rows: {len(failed_rows)}")
        print(failed_df.head(20))

        raise ValueError(
            f"{task_name}: Could not prepare all sequences. "
            f"Check failed rows here: {failed_path}"
        )

    # Build dictionaries and split arrays
    labels_dict = dict(zip(full_df["sample_id"], full_df["score"].astype(float)))
    seq_dict = dict(zip(full_df["sample_id"], full_df["esm2_sequence"].astype(str)))

    train_names = full_df.loc[full_df["split"] == "train", "sample_id"].to_numpy(dtype=object)
    val_names = full_df.loc[full_df["split"] == "val", "sample_id"].to_numpy(dtype=object)
    test_names = full_df.loc[full_df["split"] == "test", "sample_id"].to_numpy(dtype=object)

    np.save(out_dir / f"{task_name}_labels_dict.npy", labels_dict)
    np.save(out_dir / f"{task_name}_seq_dict.npy", seq_dict)
    np.save(out_dir / f"{task_name}_train_names.npy", train_names)
    np.save(out_dir / f"{task_name}_val_names.npy", val_names)
    np.save(out_dir / f"{task_name}_test_names.npy", test_names)

    full_df.to_csv(out_dir / f"{task_name}_esm2_full_table.tsv", sep="\t", index=False)

    status_counts = full_df["sequence_status"].value_counts().to_dict()

    summary = {
        "task": task_name,
        "split_dir": str(split_dir),
        "out_dir": str(out_dir),
        "total": len(full_df),
        "train": len(train_names),
        "val": len(val_names),
        "test": len(test_names),
        "unique_input_sequences": full_df["sequence"].nunique(),
        "unique_esm2_sequences": full_df["esm2_sequence"].nunique(),
        "mutation_applied": status_counts.get("mutation_applied", 0),
        "already_mutated": status_counts.get("already_mutated", 0),
        "failed": status_counts.get("failed", 0),
    }

    # Sanity checks
    all_names = np.concatenate([train_names, val_names, test_names])

    assert len(all_names) == len(set(all_names)), "Duplicate names across splits."
    assert set(all_names) == set(labels_dict.keys()), "Names and labels_dict keys do not match."
    assert set(all_names) == set(seq_dict.keys()), "Names and seq_dict keys do not match."

    if full_df["esm2_sequence"].nunique() <= 1:
        raise ValueError(
            f"{task_name}: only one unique ESM-2 sequence found. "
            "This usually means the WT sequence was not mutated correctly."
        )

    print("\nSaved files:")
    print(f"- {task_name}_labels_dict.npy")
    print(f"- {task_name}_seq_dict.npy")
    print(f"- {task_name}_train_names.npy")
    print(f"- {task_name}_val_names.npy")
    print(f"- {task_name}_test_names.npy")
    print(f"- {task_name}_esm2_full_table.tsv")

    print("\nSummary:")
    for k, v in summary.items():
        print(f"{k}: {v}")

    print("\nExample:")
    display(full_df[[
        "variant",
        "sample_id",
        "score",
        "split",
        "mutation_index_0based",
        "sequence_status",
        "sequence",
        "esm2_sequence",
    ]].head())

    return summary

In [14]:
results = []

for task_name, cfg in TASKS.items():
    split_dir = find_existing_dir(SPLIT_BASE, cfg["dir_candidates"])
    out_dir = OUT_BASE / task_name

    result = prepare_esm2_from_existing_splits(
        task_name=task_name,
        split_dir=split_dir,
        out_dir=out_dir,
        indexing=cfg["indexing"],
    )

    results.append(result)

summary_df = pd.DataFrame(results)
summary_df


Preparing UBE4B
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B
Out dir:   /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/UBE4B
Indexing:  auto
train:    752 rows loaded from train.tsv
val  :     94 rows loaded from val.tsv
test :     94 rows loaded from test.tsv

Saved files:
- UBE4B_labels_dict.npy
- UBE4B_seq_dict.npy
- UBE4B_train_names.npy
- UBE4B_val_names.npy
- UBE4B_test_names.npy
- UBE4B_esm2_full_table.tsv

Summary:
task: UBE4B
split_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B
out_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/UBE4B
total: 940
train: 752
val: 94
test: 94
unique_input_sequences: 940
unique_esm2_sequences: 940
mutation_applied: 48
already_mutated: 892
failed: 0

Example:


,variant,sample_id,score,split,mutation_index_0based,sequence_status,sequence,esm2_sequence
0,E1079L,E1079L,-0.806970,train,1079,already_mutated,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...
1,H1172D,H1172D,0.447577,train,1171,already_mutated,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...
2,P1152R,P1152R,-0.630391,train,1152,already_mutated,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...
3,D1171T,D1171T,-0.854452,train,1171,already_mutated,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...
4,K1167N,K1167N,0.647215,train,1167,already_mutated,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...,MEELSADEIRRRRLARLAGGQTSQPTTPLTSPQRENPPGPPIAASA...



Preparing GRB2
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/GRB2
Out dir:   /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/GRB2
Indexing:  auto
train:    827 rows loaded from train.tsv
val  :    103 rows loaded from val.tsv
test :    104 rows loaded from test.tsv

Saved files:
- GRB2_labels_dict.npy
- GRB2_seq_dict.npy
- GRB2_train_names.npy
- GRB2_val_names.npy
- GRB2_test_names.npy
- GRB2_esm2_full_table.tsv

Summary:
task: GRB2
split_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/GRB2
out_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/GRB2
total: 1034
train: 827
val: 103
test: 104
unique_input_sequences: 1034
unique_esm2_sequences: 1034
mutation_applied: 38
already_mutated: 996
failed: 0

Example:


,variant,sample_id,score,split,mutation_index_0based,sequence_status,sequence,esm2_sequence
0,D186M,D186M,-1.108332,train,185,already_mutated,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...
1,R177V,R177V,-0.271652,train,177,already_mutated,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...
2,N187A,N187A,-0.133569,train,187,already_mutated,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...
3,I182P,I182P,-1.031751,train,182,already_mutated,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...
4,P211I,P211I,0.006398,train,211,already_mutated,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...,MEAIAKYDFKATADDELSFKRGDILKVLNEECDQNWYKAELNGKDG...



Preparing PTEN_activity
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/PTEN_activity
Out dir:   /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/PTEN_activity
Indexing:  auto
train:   5251 rows loaded from train.tsv
val  :    656 rows loaded from val.tsv
test :    657 rows loaded from test.tsv

Saved files:
- PTEN_activity_labels_dict.npy
- PTEN_activity_seq_dict.npy
- PTEN_activity_train_names.npy
- PTEN_activity_val_names.npy
- PTEN_activity_test_names.npy
- PTEN_activity_esm2_full_table.tsv

Summary:
task: PTEN_activity
split_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/PTEN_activity
out_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/PTEN_activity
total: 6564
train: 5251
val: 656
test: 657
unique_input_sequences: 6564
unique_esm2_sequences: 6564
mutation_applied: 494
already_mutated: 6070
failed: 0

Example:


...


Preparing DLG4_abundance
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_abundance
Out dir:   /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/DLG4_abundance
Indexing:  0-based
train:   1024 rows loaded from train.tsv
val  :    128 rows loaded from val.tsv
test :    128 rows loaded from test.tsv

Saved files:
- DLG4_abundance_labels_dict.npy
- DLG4_abundance_seq_dict.npy
- DLG4_abundance_train_names.npy
- DLG4_abundance_val_names.npy
- DLG4_abundance_test_names.npy
- DLG4_abundance_esm2_full_table.tsv

Summary:
task: DLG4_abundance
split_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_abundance
out_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/DLG4_abundance
total: 1280
train: 1024
val: 128
test: 128
unique_input_sequences: 1
unique_esm2_sequences: 1280
mutation_applied: 1280
already_mutated: 0
failed: 0

Example:


,variant,sample_id,score,split,mutation_index_0based,sequence_status,sequence,esm2_sequence
0,N58S,N58S,-0.039764,train,58,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...
1,V17N,V17N,-0.066922,train,17,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNINGGEDGEGIFISFILAGGPADLSGELRKG...
2,G33P,G33P,-0.887269,train,33,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAPGPADLSGELRKG...
3,A67Y,A67Y,-0.192183,train,67,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...
4,A64V,A64V,-0.382026,train,64,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...



Preparing DLG4_binding
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_binding
Out dir:   /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/DLG4_binding
Indexing:  0-based
train:   1152 rows loaded from train.tsv
val  :    144 rows loaded from val.tsv
test :    145 rows loaded from test.tsv

Saved files:
- DLG4_binding_labels_dict.npy
- DLG4_binding_seq_dict.npy
- DLG4_binding_train_names.npy
- DLG4_binding_val_names.npy
- DLG4_binding_test_names.npy
- DLG4_binding_esm2_full_table.tsv

Summary:
task: DLG4_binding
split_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/DLG4_binding
out_dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/DLG4_binding
total: 1441
train: 1152
val: 144
test: 145
unique_input_sequences: 1
unique_esm2_sequences: 1441
mutation_applied: 1441
already_mutated: 0
failed: 0

Example:


,variant,sample_id,score,split,mutation_index_0based,sequence_status,sequence,esm2_sequence
0,L38T,L38T,-0.061865,train,38,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADTSGELRKG...
1,L12W,L12W,-0.079101,train,12,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGWGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...
2,H6W,H6W,-0.345341,train,6,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIWRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...
3,I48T,I48T,-0.924437,train,48,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...
4,A71R,A71R,-0.825313,train,71,mutation_applied,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...,PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKG...


,task,split_dir,out_dir,total,train,val,test,unique_input_sequences,unique_esm2_sequences,mutation_applied,already_mutated,failed
0,UBE4B,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,940,752,94,94,940,940,48,892,0
1,GRB2,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1034,827,103,104,1034,1034,38,996,0
2,PTEN_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,6564,5251,656,657,6564,6564,494,6070,0
3,PTEN_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,4387,3509,439,439,4387,4387,303,4084,0
4,Alpha-Amylase_expression,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7575,6060,757,758,1,7575,7575,0,0
5,Alpha-Amylase_thermostability,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7574,6059,757,758,1,7574,7574,0,0
6,Alpha-Amylase_specific_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7575,6060,757,758,1,7575,7575,0,0
7,DLG4_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1280,1024,128,128,1,1280,1280,0,0
8,DLG4_binding,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1441,1152,144,145,1,1441,1441,0,0


In [15]:
summary_path = OUT_BASE / "esm2_preprocessing_summary.tsv"
summary_df.to_csv(summary_path, sep="\t", index=False)

print("Saved summary to:")
print(summary_path)

summary_df

Saved summary to:
/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/ESM2/esm2_preprocessing_summary.tsv


,task,split_dir,out_dir,total,train,val,test,unique_input_sequences,unique_esm2_sequences,mutation_applied,already_mutated,failed
0,UBE4B,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,940,752,94,94,940,940,48,892,0
1,GRB2,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1034,827,103,104,1034,1034,38,996,0
2,PTEN_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,6564,5251,656,657,6564,6564,494,6070,0
3,PTEN_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,4387,3509,439,439,4387,4387,303,4084,0
4,Alpha-Amylase_expression,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7575,6060,757,758,1,7575,7575,0,0
5,Alpha-Amylase_thermostability,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7574,6059,757,758,1,7574,7574,0,0
6,Alpha-Amylase_specific_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,7575,6060,757,758,1,7575,7575,0,0
7,DLG4_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1280,1024,128,128,1,1280,1280,0,0
8,DLG4_binding,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,1441,1152,144,145,1,1441,1441,0,0


In [ ]:
# Check loading one of the prepared tasks
CHECK_TASK = "DLG4_binding"

check_dir = OUT_BASE / CHECK_TASK

labels_dict = np.load(check_dir / f"{CHECK_TASK}_labels_dict.npy", allow_pickle=True).item()
seq_dict = np.load(check_dir / f"{CHECK_TASK}_seq_dict.npy", allow_pickle=True).item()
train_names = np.load(check_dir / f"{CHECK_TASK}_train_names.npy", allow_pickle=True)
val_names = np.load(check_dir / f"{CHECK_TASK}_val_names.npy", allow_pickle=True)
test_names = np.load(check_dir / f"{CHECK_TASK}_test_names.npy", allow_pickle=True)

print("Task:", CHECK_TASK)
print("labels_dict:", len(labels_dict))
print("seq_dict:", len(seq_dict))
print("train:", len(train_names))
print("val:", len(val_names))
print("test:", len(test_names))

print("\nFirst examples:")
for name in train_names[:5]:
    print(name, labels_dict[name], seq_dict[name][:50], "...")

Task: DLG4_binding
labels_dict: 1441
seq_dict: 1441
train: 1152
val: 144
test: 145

First examples:
L38T -0.0618645 PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADTSGELRKGDQIL ...
L12W -0.0791008 PRRIVIHRGSTGWGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQIL ...
H6W -0.3453413 PRRIVIWRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQIL ...
I48T -0.9244372 PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQTL ...
A71R -0.8253125 PRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQIL ...


In [17]:
def check_ids_for_esm2_dataset(task_name: str):
    check_dir = OUT_BASE / task_name

    seq_dict = np.load(check_dir / f"{task_name}_seq_dict.npy", allow_pickle=True).item()
    labels_dict = np.load(check_dir / f"{task_name}_labels_dict.npy", allow_pickle=True).item()

    all_names = []

    for split in ["train", "val", "test"]:
        names = np.load(check_dir / f"{task_name}_{split}_names.npy", allow_pickle=True)
        all_names.extend(names.tolist())

    problems = []

    for name in all_names:
        uid = str(name).split("_")[0]

        if uid != name:
            problems.append((name, uid, "contains underscore"))

        if uid not in seq_dict:
            problems.append((name, uid, "uid not in seq_dict"))

        if name not in labels_dict:
            problems.append((name, uid, "name not in labels_dict"))

    if problems:
        problem_df = pd.DataFrame(problems, columns=["name", "uid_after_split", "problem"])
        display(problem_df.head(50))
        raise ValueError(f"{task_name}: found {len(problems)} ID problems.")

    print(f"{task_name}: all IDs are compatible with util_data.py")


for task_name in TASKS.keys():
    check_ids_for_esm2_dataset(task_name)

UBE4B: all IDs are compatible with util_data.py
GRB2: all IDs are compatible with util_data.py
PTEN_activity: all IDs are compatible with util_data.py
PTEN_abundance: all IDs are compatible with util_data.py
Alpha-Amylase_expression: all IDs are compatible with util_data.py
Alpha-Amylase_thermostability: all IDs are compatible with util_data.py
Alpha-Amylase_specific_activity: all IDs are compatible with util_data.py
DLG4_abundance: all IDs are compatible with util_data.py
DLG4_binding: all IDs are compatible with util_data.py


In [18]:
for task_name in TASKS.keys():
    task_dir = OUT_BASE / task_name
    print("\n" + "=" * 80)
    print(task_name)
    print("=" * 80)

    for path in sorted(task_dir.glob("*")):
        print(path.name)


UBE4B
UBE4B_esm2_full_table.tsv
UBE4B_labels_dict.npy
UBE4B_seq_dict.npy
UBE4B_test_names.npy
UBE4B_train_names.npy
UBE4B_val_names.npy

GRB2
GRB2_esm2_full_table.tsv
GRB2_labels_dict.npy
GRB2_seq_dict.npy
GRB2_test_names.npy
GRB2_train_names.npy
GRB2_val_names.npy

PTEN_activity
PTEN_activity_esm2_full_table.tsv
PTEN_activity_labels_dict.npy
PTEN_activity_seq_dict.npy
PTEN_activity_test_names.npy
PTEN_activity_train_names.npy
PTEN_activity_val_names.npy

PTEN_abundance
PTEN_abundance_esm2_full_table.tsv
PTEN_abundance_labels_dict.npy
PTEN_abundance_seq_dict.npy
PTEN_abundance_test_names.npy
PTEN_abundance_train_names.npy
PTEN_abundance_val_names.npy

Alpha-Amylase_expression
Alpha-Amylase_expression_esm2_full_table.tsv
Alpha-Amylase_expression_labels_dict.npy
Alpha-Amylase_expression_seq_dict.npy
Alpha-Amylase_expression_test_names.npy
Alpha-Amylase_expression_train_names.npy
Alpha-Amylase_expression_val_names.npy

Alpha-Amylase_thermostability
Alpha-Amylase_thermostability_esm2_full